In [15]:
import pandas as pd
test = pd.read_parquet('/Users/patrickmaloney/Documents/sports-trading/mlb-betting/data/raw/lineups/2026-02-24/lineups_013149.parquet')


In [ ]:
pd.read_parquet('/Users/patrickmaloney/Documents/sports-trading/mlb-betting/data/raw/schedules/games_2000.parquet')

In [ ]:
# =============================================
# NOTEBOOK SETUP — Run this cell first
# =============================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # goes up from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
from config.settings import DB_PATH

print("✅ Project root:", PROJECT_ROOT)
print("✅ Connecting to DuckDB (read-only):", DB_PATH)

# read_only=True = no more lock errors for analysis
con = duckdb.connect(str(DB_PATH), read_only=True)

print("✅ Connected successfully in read-only mode!")
print("Total games in silver_lineups:", 
      con.sql("SELECT COUNT(*) as total FROM silver_lineups").df().iloc[0]['total'])

Day By Day Stat Download Testing

In [2]:
# ────────────────────────────────────────────────
# Cell 1: Basic imports + one-day batting totals
# ────────────────────────────────────────────────

import pybaseball as pb
import pandas as pd

# Example: get all MLB batting stats that occurred on July 4, 2024
start_date = "2025-03-28"
# end_date   = "2024-07-04"   # same date = single day
end_date = start_date

daily_batting = pb.batting_stats_range(start_date, end_date)

print(f"Shape: {daily_batting.shape}")
display(daily_batting.head(8))          # first few rows (player-level)
print("\nColumns:", daily_batting.columns.tolist())

# Quick team totals for that day
team_totals = daily_batting.groupby('Tm').agg({
    'G': 'sum', 'PA': 'sum', 'AB': 'sum', 'R': 'sum', 'H': 'sum',
    'HR': 'sum', 'RBI': 'sum', 'BB': 'sum', 'SO': 'sum'
}).sort_values('R', ascending=False)

print("\nTeam Batting Totals on", start_date)
display(team_totals)

Shape: (311, 31)


,Name,Age,#days,Lev,Date,Tm,,Opp,G,PA,...,SH,SF,GDP,SB,CS,BA,OBP,SLG,OPS,mlbID
1,CJ Abrams,23,609,Maj-NL,"Jul 4, 2024",Washington,,New York,1,4,...,0,0,0,0,0,0.0,0.00,0.00,0.00,682928
2,Wilyer Abreu,25,609,Maj-AL,"Jul 4, 2024",Boston,@,Miami,1,3,...,0,0,0,0,1,0.0,0.00,0.00,0.00,677800
3,Willy Adames,28,609,Maj-NL,"Jul 4, 2024",Milwaukee,@,Colorado,1,4,...,0,0,0,0,0,0.0,0.00,0.00,0.00,642715
4,Riley Adams,28,609,Maj-NL,"Jul 4, 2024",Washington,,New York,1,3,...,0,0,0,0,0,1.0,1.00,1.00,2.00,656180
5,Jo Adell,25,609,Maj-AL,"Jul 4, 2024",Los Angeles,@,Oakland,1,4,...,0,0,0,1,0,0.0,0.25,0.00,0.25,666176
6,Nick Ahmed,34,609,Maj-NL,"Jul 4, 2024",San Francisco,@,Atlanta,1,4,...,0,0,0,0,0,0.0,0.00,0.00,0.00,605113
7,Ozzie Albies,27,609,Maj-NL,"Jul 4, 2024",Atlanta,,San Francisco,1,4,...,0,0,0,0,0,0.5,0.50,0.75,1.25,645277
8,Pete Alonso,29,609,Maj-NL,"Jul 4, 2024",New York,@,Washington,1,3,...,0,0,0,0,0,0.0,0.00,0.00,0.00,624413



Columns: ['Name', 'Age', '#days', 'Lev', 'Date', 'Tm', '\xa0', 'Opp', 'G', 'PA', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'BB', 'IBB', 'SO', 'HBP', 'SH', 'SF', 'GDP', 'SB', 'CS', 'BA', 'OBP', 'SLG', 'OPS', 'mlbID']

Team Batting Totals on 2024-07-04


,G,PA,AB,R,H,HR,RBI,BB,SO
Tm,,,,,,,,,
Chicago,18,76,70,14,20,4,14,5,22
Minnesota,11,41,33,12,14,1,11,6,5
Tampa Bay,11,50,41,10,16,2,10,8,6
Arizona,11,44,39,9,12,3,9,5,9
Cincinnati,12,40,36,8,9,3,8,3,10
Cleveland,10,43,36,8,12,1,8,5,7
Kansas City,11,39,36,8,10,2,7,1,4
Seattle,9,33,30,7,7,2,6,3,11
Boston,13,50,43,6,8,0,6,6,14


In [ ]:
# ────────────────────────────────────────────────
# Cell 2: Same thing but for pitching stats on a date
# ────────────────────────────────────────────────

daily_pitching = pb.pitching_stats_range(start_date, end_date)

team_pitch_totals = daily_pitching.groupby('Tm').agg({
    'G': 'sum', 'IP': 'sum', 'H': 'sum', 'R': 'sum', 'ER': 'sum',
    'HR': 'sum', 'BB': 'sum', 'SO': 'sum', 'ERA': 'mean'   # or weighted later
}).sort_values('R', ascending=True)   # lower runs allowed = better

print("\nTeam Pitching Totals Allowed on", start_date)
display(team_pitch_totals)

In [ ]:
# ────────────────────────────────────────────────
# Cell 3: One-liner function you can reuse
# ────────────────────────────────────────────────

def get_daily_team_totals(date_str: str, stat_type='batting'):
    """
    date_str: 'YYYY-MM-DD'
    stat_type: 'batting' or 'pitching'
    """
    start = end = date_str
    
    if stat_type == 'batting':
        df = pb.batting_stats_range(start, end)
        key_stats = ['G','PA','AB','R','H','2B','3B','HR','RBI','BB','SO']
    elif stat_type == 'pitching':
        df = pb.pitching_stats_range(start, end)
        key_stats = ['G','IP','H','R','ER','HR','BB','SO']
    else:
        raise ValueError("stat_type must be 'batting' or 'pitching'")
    
    if df.empty:
        print(f"No data found for {date_str}")
        return pd.DataFrame()
    
    team_df = df.groupby('Tm', as_index=False)[key_stats].sum()
    team_df = team_df.sort_values('R', ascending=(stat_type=='pitching'))
    
    print(f"{stat_type.title()} team totals for {date_str} — {len(team_df)} teams")
    return team_df

# Usage examples
display(get_daily_team_totals("2025-08-15", "batting"))
# display(get_daily_team_totals("2023-10-01", "pitching"))